# **Setup**

## 🔐 **Authenticate to Google Cloud within Colab**

Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

In [6]:
from google.colab import auth

auth.authenticate_user()

KeyboardInterrupt: ignored

## 💻 **Install Code Dependencies**
It is recommended to use the Connector alongside a library that can create connection pools, such as [SQLAlchemy](https://www.sqlalchemy.org/).
This will allow for connections to remain open and be reused, reducing connection overhead and the number of connections needed

Let's `pip install` the [Cloud SQL Python Connector](https://github.com/GoogleCloudPlatform/cloud-sql-python-connector) as well as [SQLAlchemy](https://www.sqlalchemy.org/), using the below command.

In [5]:
# install dependencies
import sys
!{sys.executable} -m pip install cloud-sql-python-connector["pymysql"] SQLAlchemy==2.0.7
!{sys.executable} -m pip install openai langchain

import google.auth
import pandas as pd

from google.cloud.sql.connector import Connector
from google.auth.transport.requests import Request
from sqlalchemy import create_engine, Table, MetaData, text, select
from sqlalchemy.orm import Session

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 47.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 33.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 2.0.10
    Uninstalling SQLAlchemy-2.0.10:
      Successfully uninstalled SQLAlchemy-2.0.10
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.2 MB/s eta

## 🐬 **Connect to a MySQL Instance**
We are now ready to connect to a MySQL instance using the Cloud SQL Python Connector! 🐍 ⭐ ☁


Let's set some parameters that are needed to connect properly to a Cloud SQL instance:
*   `INSTANCE_CONNECTION_NAME` : The connection name to your Cloud SQL Instance, takes the form `PROJECT_ID:REGION:INSTANCE_NAME`.
*   `DB_USER` : The user in which the connector will use to connect to the database.
*   `DB_PASS` : The password of the DB_USER.
*   `DB_NAME` : The name of the database on the Cloud SQL instance to connect to.

In [3]:
# initialize parameters
INSTANCE_CONNECTION_NAME = "inlaid-woods-388716:us-west1:ilkmaar" # i.e demo-project:us-central1:demo-instance
print(f"Your instance connection name is: {INSTANCE_CONNECTION_NAME}")

# IAM database user parameter (IAM user's email before the "@" sign, mysql truncates usernames)
# ex. IAM user with email "demo-user@test.com" would have database username "demo-user"

# grant Cloud SQL Client role to authenticated user
current_user = !gcloud auth list --filter=status:ACTIVE --format="value(account)"

IAM_USER = current_user[0].split("@")[0]
DB_NAME = "gameplay-data"

Your instance connection name is: inlaid-woods-388716:us-west1:ilkmaar


### ✅ **Connect to Database**


In [4]:


# initialize connector
connector = Connector()
metadata = MetaData()

# getconn now using IAM user and requiring no password with IAM Auth enabled
def getconn():
    conn = connector.connect(
      INSTANCE_CONNECTION_NAME,
      "pymysql",
      user=IAM_USER,
      db=DB_NAME,
      enable_iam_auth=True
    )
    return conn

# create connection pool
engine = create_engine(
    "mysql+pymysql://",
    creator=getconn,
)

NameError: ignored

In [ ]:
getconn()

## Test Connection

In [ ]:
# connect to connection pool
with engine.connect() as db_conn:
    # get current datetime from database
    results = db_conn.execute(text("SELECT NOW()")).fetchone()

    # output time
    print("Current time: ", results[0])

# cleanup connector
connector.close()

In [ ]:
openai_api_key="ADD_KEY_HERE_FROM_ENV"

# **GPT->SQL**

## Setup

In [2]:
from langchain import SQLDatabase, SQLDatabaseChain
from langchain.chat_models import ChatOpenAI
import pandas as pd

ModuleNotFoundError: ignored

In [ ]:
# Connect to the MySQL database
db = SQLDatabase(engine=engine, sample_rows_in_table_info=1)
llm = ChatOpenAI(temperature=0, model="gpt-4", verbose=True, openai_api_key=openai_api_key)
db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=False, use_query_checker=True, return_intermediate_steps=True, top_k=5)

def query_db(query_str):
  with engine.connect() as conn:
    return pd.read_sql_query(text(query_str), conn)

def ask_gpt(question):
  result = db_chain(question)
  answer = result['result']
  query = result['intermediate_steps'][1]
  data = query_db(query)
  
  return answer, query, data


In [ ]:
answer, query, data = ask_gpt("How many creatures are there?")

print(f"{answer}\n\nHere is the SQL query:\n{query}\n\nand here is the data:\n")
data

There are 100 creatures.

Here is the SQL query:
SELECT COUNT(*) FROM `Creatures`

and here is the data:



,COUNT(*)
0,100


In [10]:
answer, query, data = ask_gpt("What does Aisha have in her inventory?")

print(f"{answer}\n\nHere is the SQL query:\n{query}\n\nand here is the data:\n")
data

ProgrammingError: ignored

In [ ]:
answer, query, data = ask_gpt("From what location groups were resources transferred to Aisha?")

print(f"{answer}\n\nHere is the SQL query:\n{query}\n\nand here is the data:\n")
data

ProgrammingError: ignored

In [11]:
answer, query, data = ask_gpt("How many of each item have been transferred out of crafting tables?")

print(f"{answer}\n\nHere is the SQL query:\n{query}\n\nand here is the data:\n")
data

ProgrammingError: ignored

In [12]:
answer, query, data = ask_gpt("How many resources have each player crafted?")

print(f"{answer}\n\nHere is the SQL query:\n{query}\n\nand here is the data:\n")
data

ProgrammingError: ignored

In [ ]:
answer, query, data = ask_gpt("How many of each item have been transferred out of crafting tables?")

print(f"{answer}\n\nHere is the SQL query:\n{query}\n\nand here is the data:\n")
data